# 04 — PCA + K-Means Clustering
**Starbucks Customer Segmentation**

This notebook applies PCA for dimensionality reduction, then clusters customers using K-Means. Optimal K is chosen via the Elbow Method and Silhouette Score. Clusters are visualized using t-SNE.

---

## 1. Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE

sns.set_theme(style='whitegrid')

customer_df = pd.read_csv('../data/customer_features.csv', index_col='customer_id')
print('Feature table loaded. Shape:', customer_df.shape)

## 2. Scale Features

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(customer_df)
print('Features scaled. Shape:', X_scaled.shape)

## 3. PCA — Dimensionality Reduction

In [ ]:
# Fit PCA and plot explained variance
pca_full = PCA()
pca_full.fit(X_scaled)

cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(9, 4))
plt.plot(range(1, len(cumulative_variance)+1), cumulative_variance, marker='o', color='steelblue')
plt.axhline(y=0.90, color='red', linestyle='--', label='90% Variance')
plt.title('Cumulative Explained Variance by PCA Components')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.legend()
plt.tight_layout()
plt.savefig('../images/pca_variance.png', dpi=150, bbox_inches='tight')
plt.show()

# Choose components that explain 90% variance
n_components = np.argmax(cumulative_variance >= 0.90) + 1
print(f'Components to retain 90% variance: {n_components}')

In [ ]:
pca = PCA(n_components=n_components)
X_pca = pca.fit_transform(X_scaled)
print('PCA applied. Reduced shape:', X_pca.shape)

## 4. Find Optimal K — Elbow Method + Silhouette Score

In [ ]:
K_range = range(2, 11)
sse = []
silhouette_scores = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_pca)
    sse.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_pca, labels))
    print(f'K={k} | SSE={km.inertia_:.0f} | Silhouette={silhouette_score(X_pca, labels):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow curve
axes[0].plot(K_range, sse, marker='o', color='steelblue')
axes[0].set_title('Elbow Method (SSE)')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('SSE')

# Silhouette score
axes[1].plot(K_range, silhouette_scores, marker='o', color='coral')
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')

plt.tight_layout()
plt.savefig('../images/elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Final K-Means Clustering

In [ ]:
# Set optimal K based on the elbow + silhouette analysis above
OPTIMAL_K = 4  # Update this based on the plots above

kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
customer_df['cluster'] = kmeans.fit_predict(X_pca)

print(f'K-Means applied with K={OPTIMAL_K}')
print('Cluster distribution:')
print(customer_df['cluster'].value_counts().sort_index())

## 6. t-SNE Visualization

In [ ]:
print('Running t-SNE (this may take a minute)...')
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
X_tsne = tsne.fit_transform(X_pca)

tsne_df = pd.DataFrame(X_tsne, columns=['tsne_1','tsne_2'])
tsne_df['cluster'] = customer_df['cluster'].values

plt.figure(figsize=(10, 7))
sns.scatterplot(data=tsne_df, x='tsne_1', y='tsne_2',
                hue='cluster', palette='tab10', alpha=0.6, s=30)
plt.title(f't-SNE Visualization of {OPTIMAL_K} Customer Clusters')
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.legend(title='Cluster')
plt.tight_layout()
plt.savefig('../images/tsne_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Save Clustered Data

In [ ]:
customer_df.to_csv('../data/clustered_customers.csv')
print('Clustered data saved to ../data/clustered_customers.csv')